# 📊 Notebook 01 — Exploratory Data Analysis (EDA)
## Email Fraud Detection Project

**Goal:** Understand the dataset before building any models.
- Class distribution
- Message length analysis
- Top words in spam vs ham
- Text statistics

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys; sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from src.data.load_data import load_raw_data
from src.utils.helpers import dataset_summary, check_class_imbalance, plot_text_length_distribution

sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('Imports OK')

In [ ]:
# ── Load Dataset ─────────────────────────────────────────────────────────────
df = load_raw_data()
df.head(10)

In [ ]:
# ── Basic Info ───────────────────────────────────────────────────────────────
print('Shape:', df.shape)
print('\nDtypes:')
print(df.dtypes)
print('\nNull values:')
print(df.isnull().sum())
dataset_summary(df)

In [ ]:
# ── Class Distribution ────────────────────────────────────────────────────────
counts = df['label'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(counts.index, counts.values, color=['#22c55e', '#dc2626'], edgecolor='white', width=0.5)
axes[0].set_title('Class Distribution (Count)', fontsize=13)
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#22c55e', '#dc2626'], startangle=140)
axes[1].set_title('Class Distribution (%)', fontsize=13)

plt.suptitle('SMS Spam Collection — Label Distribution', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/interim/class_distribution.png', dpi=150)
plt.show()

print(f'\n⚠️  Imbalance ratio: {counts["ham"]/counts["spam"]:.1f}:1  (ham:spam)')
print('→  We will use class_weight="balanced" in our models to compensate.')

In [ ]:
# ── Text Length Analysis ──────────────────────────────────────────────────────
df['char_length'] = df['message'].str.len()
df['word_count']  = df['message'].str.split().str.len()

print(df.groupby('label')[['char_length','word_count']].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, title in zip(axes, ['char_length','word_count'],
                           ['Character Length', 'Word Count']):
    for label, color in zip(['ham','spam'],['#22c55e','#dc2626']):
        ax.hist(df[df['label']==label][col], bins=50, alpha=0.7,
                color=color, label=label, edgecolor='none')
    ax.set_title(f'{title} by Label', fontsize=12)
    ax.set_xlabel(title)
    ax.set_ylabel('Count')
    ax.legend()
plt.tight_layout()
plt.show()

print('\n💡 Insight: Spam messages tend to be LONGER than ham — they pack in more offers/urgency.')

In [ ]:
# ── Top Words: Spam vs Ham ────────────────────────────────────────────────────
from src.utils.helpers import plot_top_words

spam_texts = df[df['label']=='spam']['message'].str.lower()
ham_texts  = df[df['label']=='ham']['message'].str.lower()

plot_top_words(spam_texts, ham_texts, n=15)

In [ ]:
# ── Sample Messages ───────────────────────────────────────────────────────────
print('=== Sample SPAM messages ===')
for msg in df[df['label']=='spam']['message'].sample(3, random_state=42):
    print(f'• {msg[:120]}\n')

print('\n=== Sample HAM (Legitimate) messages ===')
for msg in df[df['label']=='ham']['message'].sample(3, random_state=42):
    print(f'• {msg[:120]}\n')

## 🔑 EDA Key Findings

1. **Imbalance**: 87% ham vs 13% spam — we must use `class_weight='balanced'` or SMOTE
2. **Length**: Spam messages are ~3x longer than ham on average
3. **Keywords**: Spam heavily uses 'free', 'call', 'txt', 'claim', 'prize', 'win'
4. **URLs/Phones**: Spam frequently contains phone numbers and URLs

→ These insights directly inform our preprocessing and feature engineering choices.